# Evaluation

In [ ]:
import os
from pathlib import Path

from dotenv import load_dotenv
from metaflow import Flow, Run

In [ ]:
PROJ_ROOT = Path.cwd().parent

In [ ]:
assert load_dotenv(dotenv_path=PROJ_ROOT.parent / ".env")

In [ ]:
import cc_churn.visualization as vzu

## About

In this notebook, the best model found from validation will be evaluated on the test split data which was not seen during validation.

As discussed in the project scope, the primary evaluation metric is the F2-score and the secondary metric is the recall. Briefly, the F2-score is important since it prioritizes recall over precision, which makes it ideal for this use-case since identifying churned customers is more important than avoiding false alarms (predicting a customer would churn when they actually did not churn). It places twice as much weight on recall as precision. This ensures the model focuses on minimizing False Negatives (predicting no churn for customers who did cancel their credit card services with the bank).

### Outputs

Artifacts of the evaluation Metaflow flow run are stored localy in the `notebooks/.metaflow/EvaluationFlow/` directory. Nothing is exported to the R2 bucket.

## User Inputs

In [ ]:
# R2 data bucket details
r2_key_train = "train_data.parquet.gzip"
r2_key_val = "validation_data.parquet.gzip"
r2_key_test = "test_data.parquet.gzip"

# datatypes for categorical and ordinal columns
dtypes_ordinals = {
    "income_category": "string[pyarrow]",
    "education_level": "string[pyarrow]",
}
dtypes_categoricals = {
    "gender": "string[pyarrow]",
    "marital_status": "string[pyarrow]",
    "card_category": "string[pyarrow]",
}

# best outputs from validation phase
best_model_name = "HistGradientBoostingClassifier"
best_experiment_run_id = "1776100277942999"

# evaluation
primary_metric_eval = "f2"
threshold_overfit = 5

In [ ]:
best_run_val = Run(f"ValidationFlow/{best_experiment_run_id}")

## Evaluation

The Metaflow flow for evaluation is shown below containing the following steps

1. `start`
   - extract best end-to-end pipeline, features, decision threshold and Metaflow run object determined in the validation phase, from the Metaflow validation flow
2. `extract`
   - load test data from R2 bucket and separate features (`X_test`) from class labels (`y_test`)
   - from best Metaflow run, extract features (`X_train`) from class labels (`y_train`) used during model validation
3. `fit`
   - train best pipeline using all training data used in model validation
4. `predict_proba`
   - predict probabilities for
     - all training data used in model validation
     - test data
5. `predict`
   - convert predicted probabilities into hard labels
6. `score`
   - score predictions
7. `gather`
   - combine true and predicted labels in test data
8. `fit_all`
   - train best pipeline using all available data in preparation for making inference predictions

In [ ]:
import json
import os

import boto3
import pandas as pd
import r2.io_utils as r2io
from cc_churn.evaluation import score_predictions
from cc_churn.mflow_utils import get_metaflow_run_artifacts
from cc_churn.scoring import get_scorers
from metaflow import FlowSpec, NBRunner, Parameter, step


class EvaluationFlow(FlowSpec):
    r2_keys = Parameter(
        name="r2_keys", help="R2 keys", default="{'test': 'E'}"
    )
    dtypes_ordinals = Parameter(
        name="dtypes_ordinals",
        help="ordinal datatypes",
        default="{'B': 'string[pyarrow]'}",
    )
    dtypes_categoricals = Parameter(
        name="dtypes_categoricals",
        help="categorical datatypes",
        default="{'G': 'string[pyarrow]'}",
    )
    best_model_name = Parameter(
        name="best_model_name",
        help="best model name from validation",
        default="LogisticRegression",
    )
    best_experiment_run_id = Parameter(
        name="best_experiment_run_id",
        help="best Metaflow Run ID",
        default="138473443",
    )

    @step
    def start(self):
        self.primary_metric_eval = "f2"
        self.metrics_list_eval = ["f2", "recall"]
        self.tpre, _, self.pipe, self.features, self.threshold, self.run = (
            get_metaflow_run_artifacts(self.best_experiment_run_id)
        )
        self.next(self.extract)

    @step
    def extract(self):
        s3_client = boto3.client(
            "s3",
            endpoint_url=(
                f"https://{os.getenv('ACCOUNT_ID')}.r2.cloudflarestorage.com"
            ),
            aws_access_key_id=os.getenv("ACCESS_KEY_ID_USER2"),
            aws_secret_access_key=os.getenv("SECRET_ACCESS_KEY_USER2"),
            region_name="auto",
        )
        df_test = (
            r2io.pandas_read_parquet_r2(
                s3_client=s3_client,
                bucket_name=os.getenv("BUCKET_NAME"),
                r2_key=json.loads(self.r2_keys)["test"],
            )
            .astype(json.loads(self.dtypes_ordinals))
            .astype(json.loads(self.dtypes_categoricals))
        )

        self.X_train = self.run.data.X
        self.y_train = self.run.data.y
        df_train = self.run.data.X.assign(is_churned=self.run.data.y)

        self.X_test = df_test.drop(columns=["is_churned"])
        self.y_test = df_test["is_churned"]

        df = pd.concat([df_train, df_test])
        self.X = df.drop(columns=["is_churned"])
        self.y = df["is_churned"]
        self.next(self.pred_proba)

    @step
    def pred_proba(self):
        self.y_train_pred_proba = pd.Series(
            self.pipe.predict_proba(self.X_train)[:, 1],
            index=self.X_train.index,
            dtype="float64[pyarrow]",
        )
        self.y_test_pred_proba = pd.Series(
            self.pipe.predict_proba(self.X_test)[:, 1],
            index=self.X_test.index,
            dtype="float64[pyarrow]",
        )
        self.next(self.predict)

    @step
    def predict(self):
        self.y_train_pred = self.y_train_pred_proba >= self.threshold
        self.y_test_pred = self.y_test_pred_proba >= self.threshold
        self.next(self.score)

    @step
    def score(self):
        self.df_scores = score_predictions(
            get_scorers(self.metrics_list_eval),
            self.y_train,
            self.y_train_pred,
            self.y_test,
            self.y_test_pred,
            self.best_model_name,
            self.primary_metric_eval,
            5,
        )
        self.next(self.gather)

    @step
    def gather(self):
        self.df_test_pred = pd.concat(
            [
                self.X_test.assign(
                    y_pred=self.y_test_pred,
                    y_pred_proba=self.y_test_pred_proba,
                ),
                self.y_test,
            ],
            axis=1,
        )
        self.next(self.fit_all)

    @step
    def fit_all(self):
        _ = self.pipe.fit(self.X, self.y)
        self.next(self.end)

    @step
    def end(self):
        pass


_ = NBRunner(EvaluationFlow, pylint=False).nbrun(
    r2_keys=json.dumps({"test": r2_key_test}),
    dtypes_ordinals=json.dumps(dtypes_ordinals),
    dtypes_categoricals=json.dumps(dtypes_categoricals),
    best_model_name=best_model_name,
    best_experiment_run_id=best_experiment_run_id,
)

Get all runs of the Metaflow evaluation flow

In [ ]:
%%time
df_eval_flow_runs = pd.DataFrame.from_records(
    [
        {
            "id": run.id,
            "started_at": run.created_at,
            "finished_at": run.finished_at,
            'tags': list(run.tags),
            'pathspec': run.pathspec,
            "finished": run.finished,
            "was_successful": run.successful,
        }
        for run in list(Flow("EvaluationFlow").runs())
    ]
)
with pd.option_context('display.max_colwidth', None):
    display(df_eval_flow_runs)

Get the most recent Metaflow evaluation flow run

In [ ]:
%%time
df_eval_runs = df_eval_flow_runs.sort_values(
    by=["finished_at"], ascending=False
).head(1)
with pd.option_context('display.max_colwidth', None):
    display(df_eval_runs)

Verify it was completed successfully

In [ ]:
assert not df_eval_runs.empty
assert df_eval_runs["finished"].squeeze() == True
assert df_eval_runs["was_successful"].squeeze() == True

Extract the following from the most recent successfully completed run of the Metaflow evaluation flow

1. run ID
2. run object
3. best decision threshold that was determined during validation and used in the evaluation flow

In [ ]:
# get ID of required Metaflow evaluation flow run
run_eval_id = df_eval_runs.query(
    "(finished == True) & (was_successful == True)"
)["id"].squeeze()

# get required Metaflow evaluation flow run object
run_eval = Run(f"EvaluationFlow/{run_eval_id}")

# extract decision threshold detremined during the validation phase, from the evaluation flow run
best_decision_threshold = run_eval.data.threshold

### Evaluate Model Performance on Test Split

Compare the evaluation metrics during model validation and evaluation

In [ ]:
%%time
df_scores_eval = (
    run_eval
    .data.df_scores.assign(split="test")
    .rename(
        columns={
            f"pct_diff_{primary_metric_eval}": "pct_diff",
            f"is_overfit_{primary_metric_eval}": "is_overfit",
            f"is_overfit_significant_{primary_metric_eval}": (
                "is_overfit_significant"
            ),
        }
    )
)
df_scores_val = (
    best_run_val.data.df_cv.assign(split="val")
    .drop(columns=["fit_time", "score_time", "estimator", "feat_group"])
    .groupby(["model_name", "split"], as_index=False)
    .agg(
        {
            f"{s}_{m}": "mean"
            for m in run_eval.data.metrics_list_eval
            for s in ["train", "test"]
        }
    )
    .assign(
        pct_diff=lambda df: (
            df[f"train_{primary_metric_eval}"]
            .sub(df[f"test_{primary_metric_eval}"])
            .abs()
            .div(df[f"train_{primary_metric_eval}"])
            .mul(100)
        ),
        is_overfit=lambda df: (
            (
                df[f"train_{primary_metric_eval}"]
                > df[f"test_{primary_metric_eval}"]
            ).astype("bool[pyarrow]")
        ),
        is_overfit_significant=lambda df: (
            (
                (df["is_overfit"] == True)
                & (df["pct_diff"] > threshold_overfit)
            ).astype("bool[pyarrow]")
        ),
    )[list(df_scores_eval)]
)
df_scores_val_eval = pd.concat(
    [df_scores_val, df_scores_eval], ignore_index=True
)
(
    df_scores_val_eval.style.set_properties(
        subset=["model_name"]
        + [f"test_{m}" for m in run_eval.data.metrics_list_eval],
        **{"background-color": "yellow", "color": "black"},
    )
    .set_properties(
        subset=[f"train_{m}" for m in run_eval.data.metrics_list_eval],
        **{"background-color": "teal", "color": "white"},
    )
    .apply(
        lambda col: [
            "background-color: darkred; color: white" if val == True else ""
            for val in col
        ],
        subset=["is_overfit_significant"],
    )
)

**Observations**

1. The scores for `f2` (primary metric during evaluatoin) and `recall` on the test data are in good agreement with scores on the validation data using ML metrics.
2. During validation, a different primary metric (`prauc`) was used to the metric used here in evaluation (`f2`). Using that metric, overfitting was significant during validation. However, using the primary metric for evaluation, overfitting is not significant during evaluation.

### Class Imbalance

Get the true and predicted values for the test split during evaluation

In [ ]:
df_test_pred = run_eval.data.df_test_pred

From these values, get the true and predicted class imbalance for the test data

In [ ]:
%%time
df_true_pred_class_imbalance = (
    (
        df_test_pred['y_pred']
        .value_counts(normalize=True)
        .rename('predicted')
        .to_frame()
    )
    .merge(
        (
            df_test_pred['is_churned']
            .value_counts(normalize=True)
            .rename('true')
            .to_frame()
        ),
        left_index=True,
        right_index=True,
    )
)
df_true_pred_class_imbalance.index = df_true_pred_class_imbalance.index.map(
    {False: 'No Churn', True: 'Churn'}
)
churn_true = df_true_pred_class_imbalance.loc['Churn']['true']
churn_pred = df_true_pred_class_imbalance.loc['Churn']['predicted']
df_true_pred_class_imbalance

**Observations**

1. The class imbalance in the test split is approximately the same as that in the training split (~84%:16%).
2. It is reassuring that the true and predicted class imbalance are close to each other.

Show the class imbalance and distribution of prediction probabilities for the test data

In [ ]:
%%time
vzu.plot_class_imbalance_proba_distribution(
    df_class_imbalance=df_true_pred_class_imbalance.rename(columns=str.title),
    df_probabilities=(df_test_pred['y_pred_proba']*100),
    ptitle1='Similar True & Predicted Churn in Test Split',
    title1_xloc=-0.3,
    ptitle2=(
        'Predicted Probabilities show Right Skew with Weak Peak Above ~90%'
    ),
    vline_label=f'Optimized Churn Cutoff ({best_decision_threshold*100:.0f}%)',
    decision_threshold=best_decision_threshold,
    subfigure_width_ratios=[1.15, 3],
    fig_size=(12, 4)
)

**Observations**

1. The dashed line shows the optimized decision threshold. Predicted probabilities below this threshold are labeled as the majority class (no churn) and those above the threshold are the minitory class (churn). As expected from the predicted class imbalance, the distribution of predicted probabilities is right-skewed and a small fraction of customers have a predicted probability above 50% (the tuned classification decision threshold).

## Conclusions

Using the best end-to-end pipeline, including the best ML model, and making predictions using the best classifier decision threshold, the validation and evaluation scores are close to each other. This is not surprising since the model has a mild class imbalance and the imbalane of the data used in validation and evaluation are close to each other.

Overfitting is insignificant. This suggests the model has successfully learned the underlying patterns rather than simply memorizing the majority class (non-churned customers) or noise in the training data. This suggests the model can generalize to customers data it has not seen before.

These two findings suggest the best model found using cost-sensitive learning is a reliable model that can maintain balanced performance by accurately identifying both the churned and non-churned credit card customers on new, unseen customer data.